# 🏠🏠🏠 Projet Kaggle : Corrélations et covariances (Variables Quantitatives) 🏠🏠🏠

## Initialisation

### Importation des bibliothèques nécessaires


In [1]:
import json

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from dash import Dash, State, callback, callback_context, dcc, html
from dash.dependencies import Input, Output

### Lecture du jeux de données d'entrainement


In [2]:
with open("../data/processed/dtype_dict.json") as f:
    dtype_dict = json.load(f)

train = pd.read_csv(
    "../data/processed/train.csv",
    delimiter=",",
    encoding="utf-8",
    index_col="Id",
    dtype=dtype_dict,
)

## Calculs et représentation des différentes corrélations

### Les corrélations entre variables quantitatives :

1. Coefficients de Pearson : Illustrent un lien linéaire entre deux variables
   quantitatives. Les prérequis pour ce calcul sont : - L’échantillon de données est aléatoire (représentatif de la population) - Les variables sont quantitatives (continues) - Les données sont associées par paires (chaque valeur x est associée à une valeur y) - Les observations sont indépendantes - Les données sont distribuées normalement - Il existe une relation linéaire entre les variables - Aucune valeur extrême n’est présente dans les données
2. Coefficients de Spearman : Mettent en lumière des liens monotones entre deux variables quantitatives. Les prérequis pour ce calcul sont :
   - L’échantillon de données est aléatoire
   - La relation entre les variables est monotone
   - Les données sont associées par paires
   - Les observations sont indépendantes
   - Il existe une relation de monotonie entre les variables
   - Les variables sont ordinales ou continues
3. Coefficients de Kendall : Très semblables aux coefficients de Spearman, ils décrivent également une relation monotone entre variables quantitatives et sont soumis aux mêmes hypothèses de départ, bien que la statistique soit légèrement différente.

Pour plus d'informations sur les formules, utilisations et explications, voici quelques liens :

- [Datalab - Corrélation de Pearson](https://datatab.fr/tutorial/pearson-correlation)
- [Datalab - Tau de Kendall](https://datatab.fr/tutorial/kendalls-tau)
- [Datalab - Coefficient de corrélation de Spearman](https://datatab.fr/tutorial/spearman-correlation)

### Calculs des corrélations


In [3]:
# Dictionnaire des corrélations
corr = {}

# Type de corrélations entre variables quantitatives
corr_quanti = ["pearson", "kendall", "spearman"]

# Calcul des corrélations après avoir retiré les colonnes non numériques
for name in corr_quanti:
    matrix = train.corr(method=name, min_periods=10, numeric_only=True)
    corr[name] = matrix

### Création d'une figure et affichage des corrélations

#### Définition d'un thème général


In [4]:
# Template personnalisé
monTheme = go.layout.Template(
    layout=dict(
        template="simple_white",
        autosize=True,
        font=dict(family="Arial", size=15, color="#000000"),
        title=dict(font=dict(size=35, family="Arial"), x=0.5),
        xaxis=dict(tickangle=-35, automargin=True),
        yaxis=dict(tickangle=-35, automargin=True),
    )
)

#### Enregistrement des principales propriétés


In [5]:
# Enregistrement du template
pio.templates["monTheme"] = monTheme

# Définition du template comme template par défaut
pio.templates.default = "monTheme"

#### Définition d'un style générique pour des boutons


In [6]:
# Même principe, style de boutons par defaut
# Ne peut pas rentrer dans les templates
styleBoutons = dict(
    bgcolor="#6B6B6B",
    bordercolor="#000000",
    borderwidth=1.5,
    direction="right",
    font_weight=700,
    showactive=True,
    type="buttons",
    x=1,
    xanchor="right",
    y=1.2,
    yanchor="top",
)

#### Définition d'un style générique pour un curseur


In [7]:
# Même chose pour le curseur
styleSlider = dict(
    active=0,
    font_color="rgba(0,0,0,0)",
    len=0.2,  # Longueur du slider
    lenmode="fraction",  # Mode de longueur du slider
    tickcolor="rgba(0,0,0,0)",
    x=0,  # Position x du slider
    xanchor="left",
    y=1.35,  # Position y du slider
    yanchor="top",
)

#### Définition d'un dictionnaire de style pour les polices (Dash)


In [8]:
mesPolices = {
    "font-size": 25,
    "font-family": "Arial",
    "font-weight": 700,
    "color": "Black",
}

#### Création des figures représentant les matrices de corrélations


In [9]:
# Initialisation de la figure Plotly et ajout des Heatmaps
heat_quanti = go.Figure()
colorscales = ["Magma", "Viridis", "Inferno"]

for idx, method in enumerate(corr_quanti):
    heatmap = go.Heatmap(
        z=corr[method].to_numpy(),
        x=corr[method].columns,
        y=corr[method].columns,
        colorscale=colorscales[idx],
        showscale=False,
        zmin=-1,
        zmax=1,
        visible=(idx == 0),
    )
    heat_quanti.add_trace(heatmap)

#### Création des boutons


In [10]:
# Paramètres des boutons
boutons_corr_quanti = [
    dict(
        label="Pearson",
        method="update",
        args=[
            {"visible": [True, False, False]},
            {"title": "Coefficients de corrélation de Pearson"},
        ],
    ),
    dict(
        label="Kendall",
        method="update",
        args=[
            {"visible": [False, True, False]},
            {"title": "Coefficients de corrélation de Kendall"},
        ],
    ),
    dict(
        label="Spearman",
        method="update",
        args=[
            {"visible": [False, False, True]},
            {"title": "Coefficients de corrélation de Spearman"},
        ],
    ),
]

#### Création du curseur avec différents seuils


In [11]:
# Paramètres du slider pour le seuil de corrélation
threshold_steps = []
for threshold in np.linspace(0, 1, 11):  # Slider de 0 à 1 par paliers de 0.1
    step = dict(
        method="update",
        args=[
            {
                "z": [
                    np.where(corr[method].abs() >= threshold, corr[method], np.nan)
                    for method in corr_quanti
                ]
            }
        ],
        label=f"{threshold:.1f}",
    )
    threshold_steps.append(step)

In [12]:
slider = [
    dict(
        **styleSlider,
        currentvalue=dict(
            prefix="Seuil : ",
            font=dict(color="#000000", weight=700),
        ),
        steps=threshold_steps,
    )
]

#### Affichage de la figure


In [13]:
# Mise à jour du layout
heat_quanti.update_layout(
    title_text="Coefficients de corrélation de Pearson",
    updatemenus=[
        dict(
            buttons=boutons_corr_quanti,
            **styleBoutons,
        )
    ],
    sliders=slider,
)

# Affichage de la figure
heat_quanti.show()

#### Sauvegarde au format HTML pour des affichages plus adaptables


In [14]:
# Exportation de la figure en fichier HTML
pio.write_html(
    heat_quanti, file="../outputs/Projet-Kaggle-Matrice-de-corrélations-quanti.html"
)

### Création d'un graphique type "nuages de points"

Ce graphique va être associé à l'heatmap pour plus d'interactivité et plus de fluidité

#### Initialisation de la figure


In [15]:
# Définition de la figure type nuages de points
scat = go.Figure(
    go.Scatter(
        x=train["LotFrontage"],
        y=train["LotFrontage"],
        mode="markers",
    )
)

#### Création des boutons


In [16]:
# Définition des boutons "x"
# X designe l'axe des abscisses
boutons_x = [
    dict(
        label=f"x - {x}",
        method="update",
        args=[
            {"x": [train[x]]},
            {"xaxis": {"title": x}},
        ],
    )
    for x in matrix.columns
]

In [17]:
# Y désigne l'axe des ordonnées
boutons_y = [
    dict(
        label=f"y - {y}",
        method="update",
        args=[
            {"y": [train[y]]},
            {"yaxis": {"title": y}},
        ],
    )
    for y in matrix.columns
]

#### Affichage de la figure


In [18]:
# Mise à jour du layout
scat.update_layout(
    title_text="Relation entre X et Y",
    xaxis_title="LotFrontage",
    yaxis_title="LotFrontage",
    updatemenus=[
        dict(
            buttons=boutons_x,
            direction="up",  # Set to 'down' or 'up' for dropdown
            showactive=True,
            x=1,
            xanchor="right",
            y=-0.25,
            yanchor="bottom",  # Custom styles specified here
            bgcolor=styleBoutons["bgcolor"],
            bordercolor=styleBoutons["bordercolor"],
            borderwidth=styleBoutons["borderwidth"],
        ),
        dict(
            buttons=boutons_y,
            direction="down",  # Set to 'down' or 'up' for dropdown
            showactive=True,
            x=0,
            xanchor="left",
            y=1.25,
            yanchor="top",  # Custom styles specified here
            bgcolor=styleBoutons["bgcolor"],
            bordercolor=styleBoutons["bordercolor"],
            borderwidth=styleBoutons["borderwidth"],
        ),
    ],
)

# Affichage de la figure
scat.show()

#### Sauvegarde au format HTML pour des affichages plus adaptables


In [19]:
# Exportation de la figure en fichier HTML
pio.write_html(scat, file="../outputs/Projet-Kaggle-graph-XY.html")

### Création d'une app Dash

#### Initialisation


In [20]:
# Initialisation de l'application Dash
app = Dash(__name__)

#### Reprise d'un graph XY simplifié


In [21]:
# Définition de la figure type nuages de points
scat = go.Figure(
    go.Scatter(
        x=train["GrLivArea"],
        y=train["SalePrice"],
        mode="markers",
    )
)

In [22]:
# Mise à jour du layout
scat.update_layout(
    title_text="Relation entre GrLivArea et SalePrice",
    xaxis_title="GrLivArea",
    yaxis_title="SalePrice",
)

#### Affichage des graphiques dans la webapp Dash

- Matrice de corrélation dynamique
- Boutons de sélection des variables quantitatives à analyser
- Graphique XY (nuages de points)


In [23]:
# Définition du layout de l'application
app.layout = html.Div(
    [
        dcc.Graph(
            id="heatmap",
            figure=heat_quanti,
            config={"responsive": True},
        ),
        html.Div(
            [
                html.Div(
                    [
                        html.Label(
                            "Sélectionner l'axe des X:",
                            style=mesPolices,
                        ),
                        dcc.Dropdown(
                            id="x-axis-dropdown",
                            options=[
                                {"label": col, "value": col} for col in matrix.columns
                            ],
                            value="GrLivArea",
                            clearable=False,
                            optionHeight=50,
                            style=mesPolices,
                        ),
                    ],
                ),
                html.Div(
                    [
                        html.Label(
                            "Sélectionner l'axe des Y:",
                            style=mesPolices,
                        ),
                        dcc.Dropdown(
                            id="y-axis-dropdown",
                            options=[
                                {"label": col, "value": col} for col in matrix.columns
                            ],
                            value="SalePrice",
                            clearable=False,
                            optionHeight=50,
                            style=mesPolices,
                        ),
                    ]
                ),
            ],
            style={"display": "flex", "justify-content": "space-around"},
        ),
        dcc.Graph(
            id="scatter",
            figure=scat,
            config={"responsive": True},
        ),
    ]
)

#### Création d'un callback pour gérer les différentes réactions :

- Suite à un clique sur la matrice de corrélations
- Suite à un changement de paramètre d'un graphique (choix de variable X ou Y)


In [24]:
# Callback pour mettre à jour le scatter plot et les dropdowns
@callback(
    [
        Output("scatter", "figure"),
        Output("x-axis-dropdown", "value"),
        Output("y-axis-dropdown", "value"),
    ],
    [
        Input("heatmap", "clickData"),
        Input("x-axis-dropdown", "value"),
        Input("y-axis-dropdown", "value"),
    ],
    [State("scatter", "figure")],
    prevent_initial_call=True,
)
def update_scatter(click_data, x_axis, y_axis, current_fig):
    ctx = callback_context

    if not ctx.triggered:
        return current_fig, x_axis, y_axis
    else:
        input_id = ctx.triggered[0]["prop_id"].split(".")[0]

    if input_id == "heatmap" and click_data:
        # Récupère les x et y sur lesquels on a cliqué
        x = click_data["points"][0]["x"]
        y = click_data["points"][0]["y"]
        scat.update_traces(x=train[x], y=train[y])
        scat.update_layout(
            title=f"Relation entre {x} et {y}",
            xaxis_title=x,
            yaxis_title=y,
        )
        return scat, x, y
    else:
        scat.update_traces(x=train[x_axis], y=train[y_axis])
        scat.update_layout(
            title_text=f"Relation entre {x_axis} et {y_axis}",
            xaxis_title=x_axis,
            yaxis_title=y_axis,
        )
        return scat, x_axis, y_axis

##### Affichage de l'app Dash


In [25]:
app.run(jupyter_mode="external", debug=True)

Dash app running on http://127.0.0.1:8050/


#### Quelques remarques sur la partie quantitative

##### Réflexions pour SalePrice

1. Encodages ordinaux  
   Les encodages ordinaux peuvent être intéressants et potentiellement utiles, notamment pour les variables GarageFinish, FireplaceQu, KitchenQual, HeatingQC, BsmtQual, ExterQual et OverallQual. En revanche, la relation est moins évidente pour d'autres variables, souvent en raison du déséquilibre dans la représentation des valeurs numériques. La présence d'hétéroscédasticité implique que des estimateurs robustes (comme les GLM) seront nécessaires pour la modélisation statistique classique. Une meilleure qualité conduit généralement à un prix plus élevé. L'absence d'un élément (comme une cheminée ou un sous-sol) semble moins avantageuse qu'une mauvaise qualité de cet élément. Des tests statistiques seront nécessaires pour confirmer cela. Il sera peut-être nécessaire de regrouper certaines modalités (par exemple, "rien" ou "mauvaise qualité") afin de rééquilibrer les données. Cette analyse pourrait être utile pour les personnes hésitant à rénover ou détruire des espaces.

2. Age de la maison lors de la vente  
   Après retraitement des dates, le graphique montre que les maisons recentes sont plus chère que les anciennes. La relation ne semble pas forcement linéaire.

3. GarageArea et GarageCars  
   Les variables GarageArea et GarageCars sont intéressantes. La relation entre la superficie du garage et le prix ne semble pas linéaire. Le découpage de GarageCars en catégories (par exemple, 3 voitures et plus) semble pertinent.

4. Fireplaces  
   La variable Fireplaces est similaire à GarageCars. Elle est pertinente pour l'interprétation (peut-être en regroupant en 2 cheminées et plus).

5. Variables de surfaces  
   Les variables de surface, telles que TotRmsAbvGrd, GrLivArea, 1stFlrSF, 2ndFlrSF (qui n'apparaît pas dans les corrélations élevées car de nombreuses maisons n'ont pas d'étage), TotalBsmtSF, FullBath, GarageArea et GarageCars sont intéressantes. En général, plus la surface est grande, plus le prix est élevé. Ces variables sont corrélées entre elles, ce qui peut introduire un risque de multicolinéarité. Plusieurs approches peuvent être envisagées pour y remédier :

   - Encodage
   - Sélection des variables les moins corrélées
   - Stacking de modèles, avec un modèle pour chaque cas :
     - Sans étage et sans sous-sol
     - Sans sous-sol
     - Sans étage
     - Avec les deux

6. LotArea  
   La taille du terrain n'apparaît pas comme fortement liée au prix de vente (SalePrice), mais je pense qu'elle mérite d'être examinée de plus près. La présence de valeurs aberrantes pourrait fausser cette relation, il est donc important de rester vigilant lors de la modélisation. Plus le terrain est grand, plus le prix semble être élevé.

##### Corrélations par construction

J'ai identifié plusieurs corrélations dues à la construction des variables :

1.  GarageCars et GarageArea  
    Une voiture occupe généralement la même superficie, ce qui explique la corrélation entre ces deux variables.

2.  TotRmsAbvGrd et GrLivArea  
    Le raisonnement est similaire à celui de GarageCars et GarageArea.

3.  Qual et Cond  
    Ces variables désignent des aspects similaires de la qualité et de la condition d'une propriété.

4.  Fireplaces et FireplacesQu_ord  
    La présence d'une cheminée détermine si la qualité est différente de zéro.

5.  PoolArea et PoolQC_ord  
    Même raisonnement que pour Fireplaces.

6.  BsmtFinSF1 et BsmtFinType1_ord, BsmtFinSF2 et BsmtFinType2_ord  
    Même raisonnement que pour Fireplaces.

7.  OverallQual et les autres variables qualifiant la qualité  
    OverallQual résume toutes les variables de qualité, ce qui explique leur corrélation logique.

##### Corrélations intéressantes

Quelques corrélations supplémentaires méritent d'être soulignées :

1.  BsmtQual_ord et DiffYearsSaleVSBuild  
    Les sous-sols des maisons plus anciennes semblent être de moins bonne qualité.

2.  TotalBsmtSF et 1stFlrSF  
    Les maisons tendent à avoir la même superficie entre le sous-sol et le premier étage en raison de la construction en étage. Cette logique peut également être appliquée à 2ndFlrSF.

3.  DiffYearsSaleVSBuild et DiffYearsRemodAddVSBuild  
    Les maisons récentes peuvent comporter des agrandissements, mais l'écart entre la construction d'origine et l'agrandissement sera très faible. En revanche, pour les maisons de plus de 60 ans, il y a presque systématiquement des agrandissements. Plus la maison est ancienne, plus l'écart entre la construction d'origine et l'agrandissement est élevé.
